# Lacrosse field detector: HRNet training on the hand labels

Finetunes PnLCalib's soccer field detectors (HRNetV2-w48: a keypoint net and a line net) on the hand-labelled frames, then measures the result **in yards** on a game it never saw.

**Run it:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

**If Colab disconnects:** reconnect and `Run all` again. Every step skips what is already done, and training resumes from the last finished epoch (a checkpoint is written after every epoch). If you mount Google Drive in step 0, the checkpoints survive a full runtime reset too; without Drive they live in `/content` and survive only a reconnect to the same runtime.

**At the end** it downloads `field_model.zip` (the two trained models, the evaluation report and pictures). Send that file back to Claude.

Pipeline (details in `docs/field_training.md`): check every label -> per-game joint fit (one lens and, on football fields, the hash distance per game; bad clicks flagged) -> training targets -> finetune -> held-out evaluation in yards.

### 0. Settings (optional: keep the run on Google Drive)

In [ ]:
USE_DRIVE = False          # set True to keep checkpoints on Google Drive (survives a full runtime reset)
KP_EPOCHS, LINE_EPOCHS, FREEZE_EPOCHS = 60, 40, 8
BATCH, ACCUM = 2, 2        # effective batch 4; drop BATCH to 1 and raise ACCUM to 4 if you see 'CUDA out of memory'

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN = '/content/drive/MyDrive/lax_field_run'
else:
    RUN = '/content/field_run'
os.makedirs(RUN, exist_ok=True)
print('run folder:', RUN)

### 1. GPU

In [ ]:
import torch, subprocess, shutil
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0), ' torch', torch.__version__)

### 2. Code (public repo, no login)

In [ ]:
%%bash
if [ -d /content/lax-vision/.git ]; then cd /content/lax-vision && git pull --ff-only -q && echo 'updated'
else git clone -q --depth 1 https://github.com/mooneybg19/lax-vision.git /content/lax-vision && echo 'cloned'; fi
cd /content/lax-vision && git log -1 --format='code version %h %s'

In [ ]:
!pip -q install opencv-python-headless scipy pyyaml
%cd /content/lax-vision

### 3. Soccer starting weights (PnLCalib v1.0.0, CC BY 4.0)
About 265 MB each. Checked by loading them, not just by size.

In [ ]:
import os, subprocess, torch
os.makedirs('/content/weights', exist_ok=True)
for name in ['SV_kp', 'SV_lines']:
    p = f'/content/weights/{name}'
    ok = False
    if os.path.exists(p):
        try: ok = len(torch.load(p, map_location='cpu', weights_only=False)) == 1839
        except Exception: ok = False
    if not ok:
        url = f'https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/{name}'
        subprocess.run(['curl', '-sSL', '--retry', '5', '-o', p, url], check=True)
        sd = torch.load(p, map_location='cpu', weights_only=False)
        assert len(sd) == 1839, f'{name}: unexpected checkpoint ({len(sd)} tensors)'
    print(name, 'ready', os.path.getsize(p) // 2**20, 'MB')

### 4. Labels
Uses `answers/field_labels.json` from the repo. If it isn't there, the cell asks you to upload the file you got from the labeling tool's **Copy all labels** (paste it into a text file named `field_labels.json`).

In [ ]:
import os, json, hashlib, shutil
LABELS = '/content/lax-vision/answers/field_labels.json'
if not os.path.exists(LABELS):
    from google.colab import files
    print('answers/field_labels.json is not in the repo: upload it now')
    up = files.upload()
    name = next(iter(up)); os.makedirs(os.path.dirname(LABELS), exist_ok=True)
    open(LABELS, 'wb').write(up[name])
docs = json.load(open(LABELS))
print(len(docs), 'documents in the export')

### 5. Check the labels and fit each game (lens, hash distance, bad clicks)
Rejected labels are listed with the reason. Then, per game: the lens coefficient k1 (with its uncertainty), the hash distance on football fields (estimate +- uncertainty, which standard it is closest to, and the value used), click error before -> after, and the frames flagged for a second look. Frames with severe flags (`low_paint_match`, `high_rms`, `poorly_constrained`) are left out of training. Skipped if already done for this exact label file.

In [ ]:
import sys, json, hashlib, os
sys.path.insert(0, '/content/lax-vision/pipeline')
sha = hashlib.sha1(open(LABELS, 'rb').read()).hexdigest()
stamp = os.path.join(RUN, 'labels.sha1')
if not (os.path.exists(os.path.join(RUN, 'prepared.json')) and os.path.exists(stamp) and open(stamp).read() == sha):
    for k in ('kp', 'line'):   # new labels -> old checkpoints no longer match the split
        for f in (f'{k}_last.pt', f'{k}_best.pt', f'{k}_log.jsonl'):
            if os.path.exists(os.path.join(RUN, f)): os.remove(os.path.join(RUN, f))
    !python pipeline/train_field.py prepare --labels {LABELS} --frames results/label_frames --out {RUN}
    open(stamp, 'w').write(sha)
else:
    print('already prepared for these labels')
gs = json.load(open(os.path.join(RUN, 'game_solve.json')))
for g, r in gs.items():
    print(f"{g:28s} k1 {r['k1']:+.4f} +- {r['k1_se']:.4f}" + (f"  hash {r.get('hash_est', r['hash']):.2f} +- {r.get('hash_se', 0):.2f} ({r.get('hash_class', '?')}), used {r['hash']:.2f}" if r['field'] == 'football' else '')
          + f"  click rms {r['before']['click_rms_px']:.2f} -> {r['after']['click_rms_px']:.2f} px  frames {r['n_frames']}  flagged {len(r['flagged_frames'])}")
    for w in r.get('warnings', []): print('   WARNING', w)
    for f, fl in list(r['flagged_frames'].items())[:8]: print('   ', f, fl)
P = json.load(open(os.path.join(RUN, 'prepared.json')))
print('train', len(P['train']), ' val', len(P['val']), ' test', len(P['test']), ' dropped', len(P['dropped_by_game_solve']))

### 6. Look at a few training targets
Red = landmark heatmaps, cyan = all painted lines, yellow = line ends. They should sit exactly on the paint. Second picture of each pair: the horizontally flipped version used in training (left and right names swap; near and far stay).

In [ ]:
import cv2, json, os, random
import field_targets as FT, field_data as FD
from IPython.display import Image, display
P = json.load(open(os.path.join(RUN, 'prepared.json')))
by = {d['frame']: d for d in P['docs']}
random.seed(0)
for f in random.sample(P['train'], min(3, len(P['train']))):
    d = by[f]; im = FD.load_image('results/label_frames', f)
    os.makedirs(os.path.join(RUN, 'targets'), exist_ok=True)
    a = os.path.join(RUN, 'targets', f + '.jpg'); b = os.path.join(RUN, 'targets', f + '_flip.jpg')
    FT.overlay(im, FT.render(d), a); FT.overlay(cv2.flip(im, 1), FT.render(d, flip=True), b)
    print(f); display(Image(a, width=900)); display(Image(b, width=900))

### 7. Train the keypoint net
About 1-3 minutes per epoch on a T4 (the first epochs, with stages 1-2 frozen, are faster). Each line shows the validation error in yards; `*` marks a new best, saved as `kp_best.pt`. Rerunning this cell after a disconnect resumes.

In [ ]:
!python pipeline/train_field.py train --kind kp --out {RUN} --frames results/label_frames --weights /content/weights --epochs {KP_EPOCHS} --freeze_epochs {FREEZE_EPOCHS} --bs {BATCH} --accum {ACCUM} --workers 2

### 8. Train the line net (used to refine the calibration)

In [ ]:
!python pipeline/train_field.py train --kind line --out {RUN} --frames results/label_frames --weights /content/weights --epochs {LINE_EPOCHS} --freeze_epochs {FREEZE_EPOCHS} --bs {BATCH} --accum {ACCUM} --workers 2

### 9. Held-out evaluation in yards
Test frames = every labelled frame of **jesuit-westview** (never trained on) plus about 15% of the dev game's frames. For each frame: detect -> calibrate -> map a grid of image points on the field to yards and compare with the label. Config **A** = keypoints only, **B** = keypoints + lines, **C** = B with the field type guessed, **D** = B with the game's lens known; **old_unet** = the earlier synthetic-data detector; **engine** (dev-game test frames only) = the current engine field fit (`field_geom`), where it produced one.

In [ ]:
!python pipeline/eval_field.py --run {RUN} --frames results/label_frames --old_unet models_cal/field_kp.pt --engine results/scituate-hanover-2024-engine --overlays 16

In [ ]:
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(os.path.join(RUN, 'eval', '*.jpg')))[:6]:
    display(Image(p, width=900))

### 10. Download the result (send `field_model.zip` to Claude)

In [ ]:
import zipfile, glob, os
from google.colab import files
z = '/content/field_model.zip'
with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as Z:
    for f in ['kp_best.pt', 'line_best.pt', 'kp_log.jsonl', 'line_log.jsonl', 'game_solve.json', 'prepared.json', 'labels.sha1']:
        p = os.path.join(RUN, f)
        if os.path.exists(p): Z.write(p, f)
    for p in glob.glob(os.path.join(RUN, 'eval', '*')): Z.write(p, os.path.join('eval', os.path.basename(p)))
print(z, os.path.getsize(z) // 2**20, 'MB')
files.download(z)